In [48]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [49]:
acn = pd.read_csv("acndata_sessions.json .csv")

In [50]:
# Working with ACN Dataset first

print(acn.info)

<bound method DataFrame.info of        _meta  end  min_kWh     site  start  _items                       _id  \
0        NaN  NaN      NaN  caltech    NaN     NaN  5bc90cb9f9af8b0d7fe77cd2   
1        NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd3   
2        NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd4   
3        NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd5   
4        NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd6   
...      ...  ...      ...      ...    ...     ...                       ...   
16299    NaN  NaN      NaN      NaN    NaN     NaN                       NaN   
16300    NaN  NaN      NaN      NaN    NaN     NaN                       NaN   
16301    NaN  NaN      NaN      NaN    NaN     NaN                       NaN   
16302    NaN  NaN      NaN      NaN    NaN     NaN                       NaN   
16303    NaN  NaN      NaN      NaN    NaN     NaN  5c2e8c38f9af8b13dab07c3b   

       

In [51]:
print(acn.columns)

Index(['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id',
       'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime',
       'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID',
       'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested',
       'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired',
       'requestedDeparture', 'userID.1'],
      dtype='object')


In [52]:
print(acn.head())

   _meta  end  min_kWh     site  start  _items                       _id  \
0    NaN  NaN      NaN  caltech    NaN     NaN  5bc90cb9f9af8b0d7fe77cd2   
1    NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd3   
2    NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd4   
3    NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd5   
4    NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd6   

   clusterID                 connectionTime                 disconnectTime  \
0       39.0  Wed, 25 Apr 2018 11:08:04 GMT  Wed, 25 Apr 2018 13:20:10 GMT   
1       39.0  Wed, 25 Apr 2018 13:45:10 GMT  Thu, 26 Apr 2018 00:56:16 GMT   
2       39.0  Wed, 25 Apr 2018 13:45:50 GMT  Wed, 25 Apr 2018 23:04:45 GMT   
3       39.0  Wed, 25 Apr 2018 14:37:06 GMT  Wed, 25 Apr 2018 23:55:34 GMT   
4       39.0  Wed, 25 Apr 2018 14:40:34 GMT  Wed, 25 Apr 2018 23:03:12 GMT   

   ... userID  userInputs WhPerMile  kWhRequested milesRequested  \
0  ...

In [53]:
# # ACN Dataset consists a lot of missing data

total_nan = acn.isna().sum()
print(total_nan)

_meta                 16304
end                   16304
min_kWh               16304
site                  16303
start                 16304
_items                16304
_id                    1305
clusterID              1305
connectionTime         1305
disconnectTime         1305
doneChargingTime       1313
kWhDelivered           1305
sessionID              1305
siteID                 1305
spaceID                1305
stationID              1305
timezone               1305
userID                14072
userInputs            16304
WhPerMile             12767
kWhRequested          12767
milesRequested        12767
minutesAvailable      12767
modifiedAt            12767
paymentRequired       12767
requestedDeparture    12767
userID.1              12767
dtype: int64


In [54]:
# percentage of missing values in each column

missing_pct = (
    acn.isnull().mean().sort_values(ascending=False) * 100
)

print(missing_pct)

_meta                 100.000000
min_kWh               100.000000
start                 100.000000
_items                100.000000
end                   100.000000
userInputs            100.000000
site                   99.993867
userID                 86.310108
requestedDeparture     78.305937
paymentRequired        78.305937
modifiedAt             78.305937
minutesAvailable       78.305937
milesRequested         78.305937
kWhRequested           78.305937
WhPerMile              78.305937
userID.1               78.305937
doneChargingTime        8.053238
timezone                8.004171
stationID               8.004171
spaceID                 8.004171
sessionID               8.004171
kWhDelivered            8.004171
disconnectTime          8.004171
connectionTime          8.004171
clusterID               8.004171
_id                     8.004171
siteID                  8.004171
dtype: float64


In [55]:
# Creating a copy for ACN Dataset for handling of missing values and manipulation

acn_clean = acn.copy()

In [56]:
# Dropping Useless Columns

drop_cols = [
    '_meta',
    '_items',
    'start',
    'end',
    'site',
    'modifiedAt'
]

acn_clean.drop(
    columns=[c for c in drop_cols if c in acn_clean.columns], inplace=True, errors='ignore')

In [57]:
# Removing duplicate user Column

if 'userID.1' in acn_clean.columns:
    acn_clean.drop(columns=['userID.1'], inplace=True)

In [58]:
# Converting datetime columns

date_cols = [
    'connectionTime',
    'disconnectTime',
    'doneChargingTime',
    'requestedDeparture'
]

for col in date_cols:
    if col in acn_clean.columns:
        acn_clean[col] = pd.to_datetime(acn_clean[col], errors='coerce')

In [59]:
# Creating missingness flags

important_cols = [
    'userID',
    'kWhRequested',
    'milesRequested',
    'minutesAvailable',
    'requestedDeparture'
]

for col in important_cols:
    if col in acn_clean.columns:
        acn_clean[f'{col}_available'] = (acn_clean[col].notna().astype(int))

In [60]:
# Handling missing values

if 'kWhRequested' in acn_clean.columns:
    acn_clean['kWhRequested'] = (
        acn_clean['kWhRequested']
        .fillna(0)
    )

if 'milesRequested' in acn_clean.columns:
    acn_clean['milesRequested'] = (
        acn_clean['milesRequested']
        .fillna(0)
    )

if 'minutesAvailable' in acn_clean.columns:
    acn_clean['minutesAvailable'] = (
        acn_clean['minutesAvailable']
        .fillna(0)
    )

if 'WhPerMile' in acn_clean.columns:
    acn_clean['WhPerMile'] = (
        acn_clean['WhPerMile']
        .fillna(
            acn_clean['WhPerMile'].median()
        )
    )

if 'paymentRequired' in acn_clean.columns:
    acn_clean['paymentRequired'] = (
        acn_clean['paymentRequired']
        .fillna(False)
        .astype(int)
    )

/var/folders/jp/mwm996tj3ts__35bq5fx0g1r0000gn/T/ipykernel_1139/2583352331.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [61]:
# Session duration calculation

acn_clean['session_duration_hr'] = (acn_clean['disconnectTime'] - acn_clean['connectionTime']).dt.total_seconds() / 3600

In [62]:
# Charging duration calculation

acn_clean['charging_duration_hr'] = (acn_clean['doneChargingTime'] - acn_clean['connectionTime']).dt.total_seconds() / 3600

acn_clean['charging_duration_hr'] = (acn_clean['charging_duration_hr'].fillna(acn_clean['session_duration_hr']))

In [63]:
# Idle time calculation

acn_clean['idle_time_hr'] = (
    acn_clean['disconnectTime'] - acn_clean['doneChargingTime']).dt.total_seconds() / 3600

acn_clean['idle_time_hr'] = (acn_clean['idle_time_hr'].fillna(0))

In [64]:
# Remove impossible records

acn_clean = acn_clean[
    (acn_clean['kWhDelivered'] > 0)
]

acn_clean = acn_clean[
    (acn_clean['session_duration_hr'] > 0)
]

In [65]:
# Time features calculation

acn_clean['hour'] = (
    acn_clean['connectionTime'].dt.hour
)

acn_clean['day_of_week'] = (
    acn_clean['connectionTime'].dt.dayofweek
)

acn_clean['month'] = (
    acn_clean['connectionTime'].dt.month
)

acn_clean['day'] = (
    acn_clean['connectionTime'].dt.day
)

acn_clean['week'] = (
    acn_clean['connectionTime'].dt.isocalendar().week
)

acn_clean['year'] = (
    acn_clean['connectionTime'].dt.year
)

acn_clean['weekend'] = (
    acn_clean['day_of_week'] >= 5
).astype(int)

In [66]:
# Charging efficiency calculation

acn_clean['charging_efficiency'] = (
    acn_clean['kWhDelivered']
    /
    acn_clean['charging_duration_hr']
)

acn_clean['charging_efficiency'] = (
    acn_clean['charging_efficiency']
    .replace([np.inf, -np.inf], np.nan)
)

In [67]:
# Utilization ratio calculation


acn_clean['utilization_ratio'] = (acn_clean['charging_duration_hr']/acn_clean['session_duration_hr'])

In [68]:
# 13. Cleaning negative values

acn_clean.loc[
    acn_clean['idle_time_hr'] < 0,
    'idle_time_hr'
] = 0

acn_clean.loc[
    acn_clean['utilization_ratio'] > 1,
    'utilization_ratio'
] = 1

In [69]:
# Removing duplicate sessions

if 'sessionID' in acn_clean.columns:
    acn_clean.drop_duplicates(
        subset='sessionID',
        inplace=True
    )

In [70]:
# 15. Resetting index

acn_clean.reset_index(
    drop=True,
    inplace=True
)

In [73]:
# Final check

print("Final Shape:", acn_clean.shape)

print("\nMissing Values:")
print(
    acn_clean.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Final Shape: (14999, 37)

Missing Values:
min_kWh                         14999
userInputs                      14999
requestedDeparture              14986
userID                          12767
doneChargingTime                    8
charging_efficiency                 2
weekend                             0
milesRequested_available            0
minutesAvailable_available          0
requestedDeparture_available        0
session_duration_hr                 0
charging_duration_hr                0
idle_time_hr                        0
year                                0
kWhRequested_available              0
day_of_week                         0
month                               0
day                                 0
week                                0
hour                                0
dtype: int64


In [74]:
# Remove useless columns

acn_clean.drop(
    columns=[
        'min_kWh',
        'userInputs',
        'requestedDeparture',
        'userID'
    ],
    inplace=True,
    errors='ignore'
)

In [75]:
# Remove invalid charging sessions

acn_clean = acn_clean[
    acn_clean['charging_duration_hr'] > 0
]

In [76]:
# Recalculate features

acn_clean['charging_efficiency'] = (
    acn_clean['kWhDelivered']
    /
    acn_clean['charging_duration_hr']
)

acn_clean['utilization_ratio'] = (
    acn_clean['charging_duration_hr']
    /
    acn_clean['session_duration_hr']
)

acn_clean['utilization_ratio'] = (
    acn_clean['utilization_ratio']
    .clip(0,1)
)

acn_clean.reset_index(
    drop=True,
    inplace=True
)

print(acn_clean.shape)
print(acn_clean.isnull().sum().sort_values(ascending=False).head())

(14979, 33)
doneChargingTime                8
_id                             0
day_of_week                     0
minutesAvailable_available      0
requestedDeparture_available    0
dtype: int64


In [82]:
acn_clean.to_csv(
    "acn_cleaned.csv",
    index=False
)